# Notebook 3: Ejecución Robusta - Errores, Configuración y Callbacks

## 🔄 Recapitulación

| Notebook | Aprendiste | Ejemplo clave |
|----------|------------|---------------|
| **N1** | Datos y estructuras básicas | `state = {"messages": [], "step": "init"}` |
| **N2** | POO, tipos, decoradores | `@tool`, `TypedDict`, `BaseModel` |

## 🎯 Objetivo de este notebook

Aprender a escribir **código robusto** que:
- No falla inesperadamente (manejo de errores)
- Usa configuración segura (variables de entorno)
- Puede "avisar" cuando pasan cosas (callbacks)

## 🧠 ¿Por qué es importante?

Cuando trabajes con LLMs (locales o APIs), muchas cosas pueden fallar:
- El modelo no está disponible
- La API rechaza tu petición
- La respuesta no tiene el formato esperado
- Se acabó tu cuota/créditos

**Código robusto = código que maneja estos problemas elegantemente.**

## 📋 Contenido

1. **Manejo de Errores** → try/except/finally
2. **Variables de Entorno** → Configuración segura
3. **Funciones como Objetos** → Fundamento para callbacks
4. **Callbacks** → "Avísame cuando pase algo"

## 📚 Librerías que usaremos

| Librería | Built-in | Para qué |
|----------|----------|----------|
| `os` | ✅ Sí | Variables de entorno |
| `dotenv` | ❌ No (pip install) | Archivos .env |

---
# PARTE 1: MANEJO DE ERRORES

## El problema

Mira este código:

In [ ]:
# ❌ CÓDIGO QUE FALLA

def dividir(a, b):
    return a / b

# Esto funciona
print(dividir(10, 2))

# Esto FALLA y detiene todo el programa
# print(dividir(10, 0))  # Descomenta para ver el error

Si descomentas la última línea, Python lanza un `ZeroDivisionError` y **todo el programa se detiene**.

En un agente, esto sería catastrófico. Imagina:
- El usuario hace una pregunta
- El agente llama a una herramienta
- La herramienta falla
- **Todo el agente muere** ❌

## La solución: try/except

```python
try:
    # Código que PUEDE fallar
    resultado = operacion_riesgosa()
except TipoDeError:
    # Qué hacer SI falla
    resultado = valor_alternativo
```

In [ ]:
# ✅ CÓDIGO QUE MANEJA EL ERROR

def dividir_seguro(a, b):
    try:
        resultado = a / b
        return resultado
    except ZeroDivisionError:
        print("⚠️ Error: No se puede dividir por cero")
        return None

# Ahora ambos funcionan sin crashear
print(f"10 / 2 = {dividir_seguro(10, 2)}")
print(f"10 / 0 = {dividir_seguro(10, 0)}")
print("\n✅ El programa continúa ejecutándose")

## Capturando el error para inspeccionarlo

Puedes guardar el error en una variable usando `as`:

In [ ]:
# Capturar el error para ver detalles

def dividir_con_info(a, b):
    try:
        return a / b
    except ZeroDivisionError as error:
        # 'error' contiene información sobre qué pasó
        print(f"❌ Error capturado: {error}")
        print(f"   Tipo: {type(error).__name__}")
        return None

dividir_con_info(10, 0)

## Tipos de errores comunes

Python tiene muchos tipos de errores (excepciones). Estos son los más comunes:

| Excepción | Cuándo ocurre | Ejemplo |
|-----------|---------------|---------|
| `ZeroDivisionError` | Dividir por cero | `10 / 0` |
| `TypeError` | Tipo incorrecto | `"hola" + 5` |
| `ValueError` | Valor incorrecto | `int("abc")` |
| `KeyError` | Clave no existe en dict | `d["clave_inexistente"]` |
| `IndexError` | Índice fuera de rango | `lista[999]` |
| `FileNotFoundError` | Archivo no existe | `open("no_existe.txt")` |
| `AttributeError` | Atributo no existe | `"hola".metodo_falso()` |
| `Exception` | Cualquier error (genérico) | Captura todo |

In [ ]:
# EJEMPLOS DE DIFERENTES ERRORES

def demostrar_errores():
    ejemplos = [
        ("TypeError", lambda: "hola" + 5),
        ("ValueError", lambda: int("abc")),
        ("KeyError", lambda: {"a": 1}["b"]),
        ("IndexError", lambda: [1, 2, 3][99]),
        ("ZeroDivisionError", lambda: 1/0),
    ]
    
    for nombre, operacion in ejemplos:
        try:
            operacion()
        except Exception as e:
            print(f"{nombre}: {e}")

demostrar_errores()

## Capturar múltiples tipos de errores

Puedes manejar diferentes errores de diferentes maneras:

In [ ]:
# Manejar diferentes errores de forma específica

def obtener_elemento(datos, indice):
    """
    Obtiene un elemento de una lista o diccionario.
    Maneja errores específicos de forma diferente.
    """
    try:
        return datos[indice]
    
    except KeyError:
        print(f"⚠️ La clave '{indice}' no existe en el diccionario")
        return None
    
    except IndexError:
        print(f"⚠️ El índice {indice} está fuera del rango de la lista")
        return None
    
    except TypeError:
        print(f"⚠️ No se puede usar '{indice}' como índice para este tipo de datos")
        return None


# Probar con diferentes casos
print("=== PRUEBAS ===")

# Caso exitoso
print(f"Lista [1,2,3] índice 1: {obtener_elemento([1,2,3], 1)}")

# IndexError
print(f"Lista [1,2,3] índice 99: {obtener_elemento([1,2,3], 99)}")

# KeyError
print(f"Dict {{'a':1}} clave 'b': {obtener_elemento({'a': 1}, 'b')}")

# TypeError
print(f"Lista [1,2,3] índice 'hola': {obtener_elemento([1,2,3], 'hola')}")

## El bloque `finally`: código que SIEMPRE se ejecuta

A veces necesitas ejecutar código sin importar si hubo error o no:

In [ ]:
# finally: siempre se ejecuta

def procesar_con_limpieza(valor):
    print(f"📥 Iniciando proceso con valor: {valor}")
    
    try:
        resultado = 100 / valor
        print(f"✅ Resultado: {resultado}")
        return resultado
    
    except ZeroDivisionError:
        print("❌ Error: división por cero")
        return None
    
    finally:
        # Este código SIEMPRE se ejecuta
        print("🧹 Limpieza completada (finally)")
        print("-" * 40)


# Caso exitoso
procesar_con_limpieza(5)

# Caso con error
procesar_con_limpieza(0)

## `raise`: Lanzar tus propios errores

A veces TÚ quieres generar un error intencionalmente:

In [ ]:
# raise: lanzar errores intencionalmente

def validar_edad(edad):
    """Valida que la edad sea un número positivo razonable."""
    
    if not isinstance(edad, int):
        raise TypeError(f"La edad debe ser un entero, no {type(edad).__name__}")
    
    if edad < 0:
        raise ValueError("La edad no puede ser negativa")
    
    if edad > 150:
        raise ValueError("La edad no puede ser mayor a 150 años")
    
    return True


# Probar validaciones
casos = [25, -5, 200, "treinta"]

for edad in casos:
    try:
        validar_edad(edad)
        print(f"✅ Edad {edad} es válida")
    except (TypeError, ValueError) as e:
        print(f"❌ Edad {edad}: {e}")

## 🔗 Conexión con LangChain

En LangChain verás exactamente estos patrones:

```python
# Código real de LangChain (referencia)
from langchain_ollama import ChatOllama

llm = ChatOllama(model="llama3")

try:
    response = llm.invoke("Hola")
    print(response.content)
    
except Exception as e:
    print(f"Error al llamar al modelo: {e}")
    # Quizás intentar con otro modelo
    # O devolver un mensaje de error al usuario
```

**Errores comunes en LangChain:**
- `ConnectionError`: El modelo local no está corriendo
- `AuthenticationError`: API key inválida
- `RateLimitError`: Demasiadas peticiones
- `OutputParserException`: La respuesta no tiene el formato esperado

In [ ]:
# SIMULACIÓN: Patrón de reintento para LLMs

import random
import time

def simular_llm_inestable(prompt):
    """
    Simula un LLM que a veces falla (como en la vida real).
    Falla ~50% de las veces.
    """
    if random.random() < 0.5:
        raise ConnectionError("No se pudo conectar al modelo")
    return f"Respuesta a: {prompt}"


def invocar_con_reintentos(prompt, max_intentos=3):
    """
    Intenta llamar al LLM varias veces antes de rendirse.
    Este patrón es MUY común en producción.
    """
    for intento in range(1, max_intentos + 1):
        try:
            print(f"  Intento {intento}/{max_intentos}...")
            resultado = simular_llm_inestable(prompt)
            print(f"  ✅ Éxito en intento {intento}")
            return resultado
        
        except ConnectionError as e:
            print(f"  ⚠️ Falló: {e}")
            if intento < max_intentos:
                print(f"  ⏳ Esperando antes de reintentar...")
                time.sleep(0.5)  # Esperar antes de reintentar
    
    print(f"  ❌ Falló después de {max_intentos} intentos")
    return None


# Probar el patrón de reintento
print("=== PRUEBA DE REINTENTOS ===")
resultado = invocar_con_reintentos("¿Cuál es la capital de Francia?")
print(f"\nResultado final: {resultado}")

---
# PARTE 2: VARIABLES DE ENTORNO

## El problema: secretos en el código

Mira este código **PELIGROSO**:

```python
# ❌ NUNCA HAGAS ESTO
api_key = "sk-abc123secretkey456"
llm = ChatOpenAI(api_key=api_key)
```

**¿Por qué es malo?**
1. Si subes el código a GitHub, todos ven tu API key
2. Si compartes el código, compartes tus secretos
3. Si cambias la key, debes modificar el código

## La solución: Variables de entorno

Las **variables de entorno** son valores que existen en el sistema operativo, fuera de tu código.

In [ ]:
# Librería os: acceso al sistema operativo
import os

# Ver algunas variables de entorno que ya existen
print("=== VARIABLES DE ENTORNO DEL SISTEMA ===")
print(f"Usuario: {os.environ.get('USER', 'No definido')}")
print(f"Home: {os.environ.get('HOME', 'No definido')}")
print(f"Shell: {os.environ.get('SHELL', 'No definido')}")
print(f"Path: {os.environ.get('PATH', 'No definido')[:50]}...")  # Truncado

## `os.environ`: El diccionario de variables de entorno

`os.environ` funciona como un diccionario de Python:

In [ ]:
import os

# Establecer una variable de entorno
os.environ["MI_VARIABLE"] = "mi_valor_secreto"
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

# Leer variables de entorno
print(f"MI_VARIABLE: {os.environ['MI_VARIABLE']}")
print(f"OLLAMA_HOST: {os.environ['OLLAMA_HOST']}")

# ⚠️ Leer una variable que NO existe causa KeyError
try:
    print(os.environ["VARIABLE_INEXISTENTE"])
except KeyError:
    print("❌ VARIABLE_INEXISTENTE no existe")

## `.get()`: Leer de forma segura

Usa `.get()` para evitar errores cuando la variable no existe:

In [ ]:
import os

# .get() con valor por defecto (patrón recomendado)

# Si existe, devuelve el valor
ollama_host = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
print(f"OLLAMA_HOST: {ollama_host}")

# Si NO existe, devuelve el valor por defecto
openai_key = os.environ.get("OPENAI_API_KEY", "no-configurada")
print(f"OPENAI_API_KEY: {openai_key}")

# Verificar si una variable está configurada
if os.environ.get("OPENAI_API_KEY"):
    print("✅ API key configurada")
else:
    print("⚠️ API key NO configurada")

## Archivos `.env`: Configuración persistente

En lugar de escribir `export VARIABLE=valor` cada vez, usamos archivos `.env`:

```
# Archivo: .env (en la raíz del proyecto)
OPENAI_API_KEY=sk-tu-api-key-aqui
OLLAMA_HOST=http://localhost:11434
DEBUG=true
```

**IMPORTANTE:** El archivo `.env` debe estar en `.gitignore` para no subirlo a GitHub.

Para leer archivos `.env` usamos la librería `python-dotenv`:

In [ ]:
# Primero, simulemos crear un archivo .env

contenido_env = """# Archivo de configuración (NO subir a git)
OPENAI_API_KEY=sk-demo-key-12345
OLLAMA_HOST=http://localhost:11434
MODELO_DEFAULT=llama3
DEBUG=true
"""

# Crear el archivo
with open(".env.ejemplo", "w") as f:
    f.write(contenido_env)

print("✅ Archivo .env.ejemplo creado")
print("\nContenido:")
print(contenido_env)

In [ ]:
# USAR python-dotenv para cargar el archivo .env
# (pip install python-dotenv si no está instalado)

try:
    from dotenv import load_dotenv
    
    # Cargar variables desde el archivo
    load_dotenv(".env.ejemplo")
    
    # Ahora las variables están en os.environ
    import os
    print("=== VARIABLES CARGADAS DESDE .env ===")
    print(f"OPENAI_API_KEY: {os.environ.get('OPENAI_API_KEY', 'No cargada')}")
    print(f"OLLAMA_HOST: {os.environ.get('OLLAMA_HOST', 'No cargada')}")
    print(f"MODELO_DEFAULT: {os.environ.get('MODELO_DEFAULT', 'No cargada')}")
    print(f"DEBUG: {os.environ.get('DEBUG', 'No cargada')}")
    
except ImportError:
    print("⚠️ python-dotenv no está instalado")
    print("   Instálalo con: pip install python-dotenv")
    print("\n📝 Simulando carga manual:")
    
    # Simulación sin la librería
    import os
    os.environ["OPENAI_API_KEY"] = "sk-demo-key-12345"
    os.environ["OLLAMA_HOST"] = "http://localhost:11434"
    print(f"OPENAI_API_KEY: {os.environ['OPENAI_API_KEY']}")

## Patrón: Configuración centralizada

En proyectos reales, es buena práctica tener un módulo de configuración:

In [ ]:
# PATRÓN: Módulo de configuración

import os

class Config:
    """
    Configuración centralizada del proyecto.
    Lee de variables de entorno con valores por defecto.
    """
    
    # API Keys
    OPENAI_API_KEY: str = os.environ.get("OPENAI_API_KEY", "")
    
    # Configuración de modelos locales
    OLLAMA_HOST: str = os.environ.get("OLLAMA_HOST", "http://localhost:11434")
    MODELO_DEFAULT: str = os.environ.get("MODELO_DEFAULT", "llama3")
    
    # Configuración general
    DEBUG: bool = os.environ.get("DEBUG", "false").lower() == "true"
    MAX_REINTENTOS: int = int(os.environ.get("MAX_REINTENTOS", "3"))
    
    @classmethod
    def validar(cls):
        """Verifica que la configuración esencial esté presente."""
        errores = []
        
        # Verificar configuración requerida
        if not cls.OLLAMA_HOST:
            errores.append("OLLAMA_HOST no configurado")
        
        if errores:
            for error in errores:
                print(f"❌ {error}")
            return False
        
        print("✅ Configuración válida")
        return True
    
    @classmethod
    def mostrar(cls):
        """Muestra la configuración actual (ocultando secretos)."""
        print("=== CONFIGURACIÓN ACTUAL ===")
        print(f"OPENAI_API_KEY: {'***' + cls.OPENAI_API_KEY[-4:] if cls.OPENAI_API_KEY else 'No configurada'}")
        print(f"OLLAMA_HOST: {cls.OLLAMA_HOST}")
        print(f"MODELO_DEFAULT: {cls.MODELO_DEFAULT}")
        print(f"DEBUG: {cls.DEBUG}")
        print(f"MAX_REINTENTOS: {cls.MAX_REINTENTOS}")


# Usar la configuración
Config.mostrar()
print()
Config.validar()

## 🔗 Conexión con LangChain

LangChain lee automáticamente ciertas variables de entorno:

```python
# Si OPENAI_API_KEY está en el entorno, no necesitas pasarla
import os
os.environ["OPENAI_API_KEY"] = "tu-api-key"

from langchain_openai import ChatOpenAI
llm = ChatOpenAI()  # Automáticamente usa OPENAI_API_KEY

# Para Ollama local, puedes configurar el host
os.environ["OLLAMA_HOST"] = "http://localhost:11434"

from langchain_ollama import ChatOllama
llm = ChatOllama(model="llama3")  # Usa OLLAMA_HOST
```

---
# PARTE 3: FUNCIONES COMO OBJETOS

## Repaso: ¿Qué es una función?

En el Notebook 1 aprendiste a crear funciones:

In [ ]:
# Repaso: función básica

def saludar(nombre):
    """Devuelve un saludo."""
    return f"¡Hola, {nombre}!"

# Llamar a la función
resultado = saludar("María")
print(resultado)

## Concepto clave: Las funciones SON objetos

En Python, **todo es un objeto**, incluyendo las funciones. Esto significa que:
- Puedes guardar funciones en variables
- Puedes pasar funciones como argumentos
- Puedes devolver funciones desde otras funciones

In [ ]:
# Las funciones son objetos

def sumar(a, b):
    return a + b

# La función tiene atributos, como cualquier objeto
print(f"Nombre de la función: {sumar.__name__}")
print(f"Tipo: {type(sumar)}")
print(f"Documentación: {sumar.__doc__}")

# ¿Recuerdas los decoradores del Notebook 2?
# Funcionan porque las funciones son objetos

In [ ]:
# GUARDAR funciones en variables

def sumar(a, b):
    return a + b

def restar(a, b):
    return a - b

def multiplicar(a, b):
    return a * b

# Guardar funciones en variables
# Nota: SIN paréntesis - no estamos llamando a la función
operacion = sumar

# Ahora 'operacion' es la función sumar
print(f"operacion(5, 3) = {operacion(5, 3)}")

# Cambiar la operación
operacion = multiplicar
print(f"operacion(5, 3) = {operacion(5, 3)}")

In [ ]:
# GUARDAR funciones en estructuras de datos

def sumar(a, b):
    return a + b

def restar(a, b):
    return a - b

def multiplicar(a, b):
    return a * b

def dividir(a, b):
    return a / b if b != 0 else "Error: división por cero"

# Diccionario de operaciones
operaciones = {
    "+": sumar,
    "-": restar,
    "*": multiplicar,
    "/": dividir,
}

# Usar el diccionario para ejecutar operaciones
def calcular(a, operador, b):
    if operador not in operaciones:
        return f"Operador '{operador}' no soportado"
    
    funcion = operaciones[operador]  # Obtener la función
    return funcion(a, b)             # Llamar a la función

# Probar
print(f"10 + 5 = {calcular(10, '+', 5)}")
print(f"10 - 5 = {calcular(10, '-', 5)}")
print(f"10 * 5 = {calcular(10, '*', 5)}")
print(f"10 / 5 = {calcular(10, '/', 5)}")

## Pasar funciones como argumentos

Puedes pasar una función a otra función como argumento:

In [ ]:
# PASAR funciones como argumentos

def aplicar_a_lista(lista, funcion):
    """
    Aplica una función a cada elemento de la lista.
    
    Args:
        lista: Lista de elementos
        funcion: Función a aplicar a cada elemento
    """
    resultados = []
    for elemento in lista:
        resultado = funcion(elemento)  # Llamar a la función recibida
        resultados.append(resultado)
    return resultados


# Definir algunas funciones simples
def duplicar(x):
    return x * 2

def al_cuadrado(x):
    return x ** 2

def es_par(x):
    return x % 2 == 0


# Usar aplicar_a_lista con diferentes funciones
numeros = [1, 2, 3, 4, 5]

print(f"Original: {numeros}")
print(f"Duplicados: {aplicar_a_lista(numeros, duplicar)}")
print(f"Al cuadrado: {aplicar_a_lista(numeros, al_cuadrado)}")
print(f"¿Es par?: {aplicar_a_lista(numeros, es_par)}")

### Funciones lambda: Funciones anónimas en una línea

Para funciones simples, puedes usar `lambda`:

In [ ]:
# Lambda: funciones anónimas

# Función normal
def duplicar(x):
    return x * 2

# Equivalente con lambda
duplicar_lambda = lambda x: x * 2

# Son equivalentes
print(f"duplicar(5) = {duplicar(5)}")
print(f"duplicar_lambda(5) = {duplicar_lambda(5)}")

# Lambda es útil para funciones pequeñas que no reutilizarás
numeros = [1, 2, 3, 4, 5]

# En lugar de definir una función, usamos lambda directamente
print(f"\nTriplicados: {aplicar_a_lista(numeros, lambda x: x * 3)}")
print(f"Más 10: {aplicar_a_lista(numeros, lambda x: x + 10)}")

## 🔗 ¿Por qué esto importa para agentes?

En el Notebook 2 viste que los decoradores modifican funciones. Ahora entiendes **cómo** lo hacen:

```python
# Del Notebook 2
@tool
def search(query: str) -> str:
    """Busca información."""
    return results

# Es equivalente a:
def search(query: str) -> str:
    """Busca información."""
    return results

search = tool(search)  # ← Pasar función como argumento
```

El decorador `@tool` recibe la función `search` como argumento y devuelve una versión modificada.

---
# PARTE 4: CALLBACKS

## ¿Qué es un callback?

Un **callback** es una función que le pasas a otra función para que la llame cuando algo ocurra.

Piensa en esto:
> "Oye, cuando termines de procesar, **llámame** y avísame."

El "llámame" es el callback.

In [ ]:
# CALLBACK BÁSICO: "Avísame cuando termines"

def procesar_datos(datos, callback_al_terminar):
    """
    Procesa los datos y luego llama al callback.
    
    Args:
        datos: Lista de datos a procesar
        callback_al_terminar: Función a llamar cuando termine
    """
    print("📥 Iniciando procesamiento...")
    
    # Simular procesamiento
    resultado = [x * 2 for x in datos]
    
    print("✅ Procesamiento completado")
    
    # Llamar al callback con el resultado
    callback_al_terminar(resultado)


# Definir nuestro callback
def mi_callback(resultado):
    print(f"📞 Callback recibido! Resultado: {resultado}")


# Usar la función con el callback
procesar_datos([1, 2, 3, 4, 5], mi_callback)

## Callbacks múltiples: eventos diferentes

Puedes tener callbacks para diferentes eventos:

In [ ]:
# MÚLTIPLES CALLBACKS: Diferentes eventos

def procesar_con_eventos(datos, on_inicio=None, on_progreso=None, on_fin=None, on_error=None):
    """
    Procesa datos con callbacks para diferentes eventos.
    
    Args:
        datos: Lista a procesar
        on_inicio: Callback al iniciar
        on_progreso: Callback en cada paso (recibe índice y total)
        on_fin: Callback al terminar (recibe resultado)
        on_error: Callback si hay error (recibe el error)
    """
    try:
        # Evento: inicio
        if on_inicio:
            on_inicio()
        
        resultados = []
        total = len(datos)
        
        for i, dato in enumerate(datos):
            # Evento: progreso
            if on_progreso:
                on_progreso(i + 1, total)
            
            # Simular posible error
            if dato == "error":
                raise ValueError("Dato inválido encontrado")
            
            resultados.append(dato * 2)
        
        # Evento: fin exitoso
        if on_fin:
            on_fin(resultados)
        
        return resultados
    
    except Exception as e:
        # Evento: error
        if on_error:
            on_error(e)
        return None


# Definir callbacks
def al_iniciar():
    print("🚀 Proceso iniciado")

def al_progresar(actual, total):
    porcentaje = (actual / total) * 100
    print(f"   📊 Progreso: {actual}/{total} ({porcentaje:.0f}%)")

def al_terminar(resultado):
    print(f"✅ Proceso completado. Resultado: {resultado}")

def al_fallar(error):
    print(f"❌ Error: {error}")


# Probar con caso exitoso
print("=== CASO EXITOSO ===")
procesar_con_eventos(
    [1, 2, 3, 4, 5],
    on_inicio=al_iniciar,
    on_progreso=al_progresar,
    on_fin=al_terminar,
    on_error=al_fallar
)

In [ ]:
# Probar con caso que falla
print("=== CASO CON ERROR ===")
procesar_con_eventos(
    [1, 2, "error", 4, 5],
    on_inicio=al_iniciar,
    on_progreso=al_progresar,
    on_fin=al_terminar,
    on_error=al_fallar
)

## Patrón de clase con callbacks (como LangChain)

LangChain usa clases para definir callbacks. Veamos un ejemplo simplificado:

In [ ]:
# PATRÓN DE CALLBACKS CON CLASES (estilo LangChain)

from typing import Optional


class CallbackHandler:
    """
    Clase base para manejar callbacks.
    Similar a BaseCallbackHandler de LangChain.
    """
    
    def on_llm_start(self, prompt: str):
        """Llamado cuando el LLM empieza a procesar."""
        pass
    
    def on_llm_end(self, response: str):
        """Llamado cuando el LLM termina."""
        pass
    
    def on_llm_error(self, error: Exception):
        """Llamado si el LLM falla."""
        pass
    
    def on_tool_start(self, tool_name: str, input_text: str):
        """Llamado cuando una herramienta empieza."""
        pass
    
    def on_tool_end(self, output: str):
        """Llamado cuando una herramienta termina."""
        pass


class LoggingCallback(CallbackHandler):
    """
    Callback que imprime todo lo que pasa.
    Hereda de CallbackHandler y sobrescribe los métodos.
    """
    
    def on_llm_start(self, prompt: str):
        print(f"🤖 LLM iniciado con: '{prompt[:50]}...'")
    
    def on_llm_end(self, response: str):
        print(f"🤖 LLM respondió: '{response[:50]}...'")
    
    def on_llm_error(self, error: Exception):
        print(f"❌ LLM falló: {error}")
    
    def on_tool_start(self, tool_name: str, input_text: str):
        print(f"🔧 Tool '{tool_name}' iniciada con: '{input_text}'")
    
    def on_tool_end(self, output: str):
        print(f"🔧 Tool completada: '{output}'")


class MetricsCallback(CallbackHandler):
    """
    Callback que registra métricas.
    """
    
    def __init__(self):
        self.llm_calls = 0
        self.tool_calls = 0
        self.errors = 0
    
    def on_llm_start(self, prompt: str):
        self.llm_calls += 1
    
    def on_tool_start(self, tool_name: str, input_text: str):
        self.tool_calls += 1
    
    def on_llm_error(self, error: Exception):
        self.errors += 1
    
    def get_metrics(self):
        return {
            "llm_calls": self.llm_calls,
            "tool_calls": self.tool_calls,
            "errors": self.errors
        }


print("✅ Clases de callback definidas")

In [ ]:
# SIMULACIÓN: LLM con callbacks

import random
from typing import List


class SimuladorLLM:
    """
    Simula un LLM con soporte para callbacks.
    """
    
    def __init__(self, callbacks: List[CallbackHandler] = None):
        self.callbacks = callbacks or []
    
    def invoke(self, prompt: str) -> str:
        """Simula una llamada al LLM."""
        
        # Notificar inicio
        for cb in self.callbacks:
            cb.on_llm_start(prompt)
        
        try:
            # Simular fallo ocasional
            if random.random() < 0.2:
                raise ConnectionError("Fallo de conexión simulado")
            
            # Simular respuesta
            response = f"Respuesta simulada para: {prompt}"
            
            # Notificar fin exitoso
            for cb in self.callbacks:
                cb.on_llm_end(response)
            
            return response
        
        except Exception as e:
            # Notificar error
            for cb in self.callbacks:
                cb.on_llm_error(e)
            raise


# Crear callbacks
logger = LoggingCallback()
metrics = MetricsCallback()

# Crear LLM con callbacks
llm = SimuladorLLM(callbacks=[logger, metrics])

# Hacer varias llamadas
print("=== SIMULACIÓN DE LLAMADAS AL LLM ===\n")

for i in range(5):
    print(f"--- Llamada {i+1} ---")
    try:
        resultado = llm.invoke(f"Pregunta número {i+1}")
    except:
        pass  # El callback ya manejó el error
    print()

# Ver métricas
print("=== MÉTRICAS FINALES ===")
print(metrics.get_metrics())

## 🔗 Conexión con LangChain

Lo que acabas de ver es **exactamente** el patrón de LangChain:

```python
# Código real de LangChain (referencia)
from langchain.callbacks.base import BaseCallbackHandler
from langchain_ollama import ChatOllama

class MiCallback(BaseCallbackHandler):
    def on_llm_start(self, serialized, prompts, **kwargs):
        print(f"LLM iniciado con: {prompts}")
    
    def on_llm_end(self, response, **kwargs):
        print(f"LLM terminó: {response}")

# Usar el callback
llm = ChatOllama(model="llama3", callbacks=[MiCallback()])
llm.invoke("Hola")
```

**Callbacks comunes en LangChain:**
- `on_llm_start` / `on_llm_end`: Inicio/fin de LLM
- `on_chat_model_start`: Inicio de chat
- `on_tool_start` / `on_tool_end`: Uso de herramientas
- `on_chain_start` / `on_chain_end`: Ejecución de chains

---
# INTEGRACIÓN: Combinando todo

Veamos un ejemplo que combina errores, configuración y callbacks:

In [ ]:
# EJEMPLO INTEGRADO: Simulador de agente robusto

import os
import random
from typing import List, Optional
from datetime import datetime


# === CONFIGURACIÓN ===
class AgentConfig:
    MODEL_NAME: str = os.environ.get("MODEL_NAME", "llama3")
    MAX_RETRIES: int = int(os.environ.get("MAX_RETRIES", "3"))
    DEBUG: bool = os.environ.get("DEBUG", "true").lower() == "true"


# === CALLBACKS ===
class AgentCallback:
    def on_start(self, prompt: str): pass
    def on_retry(self, attempt: int, max_attempts: int, error: Exception): pass
    def on_success(self, response: str): pass
    def on_failure(self, error: Exception): pass


class VerboseCallback(AgentCallback):
    """Callback que imprime todo."""
    
    def on_start(self, prompt: str):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] 🚀 Iniciando: '{prompt}'")
    
    def on_retry(self, attempt: int, max_attempts: int, error: Exception):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] 🔄 Reintento {attempt}/{max_attempts}: {error}")
    
    def on_success(self, response: str):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ✅ Éxito: '{response[:50]}'")
    
    def on_failure(self, error: Exception):
        print(f"[{datetime.now().strftime('%H:%M:%S')}] ❌ Falló definitivamente: {error}")


# === AGENTE ===
class SimulatedAgent:
    """
    Agente simulado con manejo de errores, configuración y callbacks.
    Combina todo lo aprendido en este notebook.
    """
    
    def __init__(self, callbacks: List[AgentCallback] = None):
        self.config = AgentConfig()
        self.callbacks = callbacks or []
        
        if self.config.DEBUG:
            print(f"Agent inicializado: model={self.config.MODEL_NAME}, retries={self.config.MAX_RETRIES}")
    
    def _notify(self, event: str, **kwargs):
        """Notifica a todos los callbacks."""
        for cb in self.callbacks:
            method = getattr(cb, event, None)
            if method:
                method(**kwargs)
    
    def _simulate_llm_call(self, prompt: str) -> str:
        """Simula una llamada al LLM (falla 50% de las veces)."""
        if random.random() < 0.5:
            raise ConnectionError("Error de conexión al modelo")
        return f"[{self.config.MODEL_NAME}] Respuesta a: {prompt}"
    
    def invoke(self, prompt: str) -> Optional[str]:
        """
        Invoca el agente con reintentos y callbacks.
        
        Combina:
        - try/except (manejo de errores)
        - Configuración (número de reintentos)
        - Callbacks (notificaciones)
        """
        self._notify("on_start", prompt=prompt)
        
        last_error = None
        
        for attempt in range(1, self.config.MAX_RETRIES + 1):
            try:
                response = self._simulate_llm_call(prompt)
                self._notify("on_success", response=response)
                return response
            
            except Exception as e:
                last_error = e
                if attempt < self.config.MAX_RETRIES:
                    self._notify("on_retry", attempt=attempt, max_attempts=self.config.MAX_RETRIES, error=e)
        
        self._notify("on_failure", error=last_error)
        return None


# === USAR EL AGENTE ===
print("=== SIMULACIÓN DE AGENTE ROBUSTO ===\n")

agent = SimulatedAgent(callbacks=[VerboseCallback()])

for i in range(3):
    print(f"\n--- Consulta {i+1} ---")
    result = agent.invoke(f"¿Cuál es la capital de España?")
    if result:
        print(f"    Resultado obtenido: {result}")
    else:
        print(f"    Sin resultado (todos los reintentos fallaron)")

---
# 📝 Resumen del Notebook 3

## Lo que aprendiste

| Concepto | Para qué | Patrón LangChain |
|----------|----------|------------------|
| **try/except** | Manejar errores sin crashear | `try: llm.invoke() except:` |
| **Variables de entorno** | Configuración segura | `os.environ["OPENAI_API_KEY"]` |
| **Funciones como objetos** | Base de decoradores/callbacks | Fundamento |
| **Callbacks** | Recibir notificaciones | `callbacks=[MiCallback()]` |

## Código real que ahora entiendes

```python
# Configuración
import os
from dotenv import load_dotenv
load_dotenv()

# Callback personalizado
class MiCallback(BaseCallbackHandler):
    def on_llm_end(self, response, **kwargs):
        print(f"Tokens usados: {response.llm_output}")

# LLM con manejo de errores y callbacks
llm = ChatOllama(model="llama3", callbacks=[MiCallback()])

try:
    response = llm.invoke("Hola")
except Exception as e:
    print(f"Error: {e}")
```

## Siguiente paso

**Notebook 4**: Patrones de flujo (generators, streaming, async/await)